In [23]:
import os
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document

## 1. Rebuild the Same Configuration ##

In [24]:
project_root = Path.cwd()
print(f"Project root: {project_root}")

collection_name = "demo_collection"
persist_directory = project_root / "chroma_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Project root: d:\LangChain Fundamentals\Retrievers in LangChain v1
Collection name: demo_collection
Persist directory: d:\LangChain Fundamentals\Retrievers in LangChain v1\chroma_db


In [25]:
client = chromadb.PersistentClient(path=str(persist_directory))

print("Connected to the existing Chroma collection.")

Connected to the existing Chroma collection.


In [26]:
data_chromadb = client.get_collection(name="demo_collection").get(include=["metadatas", "documents", "embeddings"])
print(f"Number of records in the collection: {len(data_chromadb['ids'])}")

Number of records in the collection: 27


## 2. Add Small Display Helpers ##

In [27]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

In [28]:
print_stored_documents(data_chromadb)

Total documents in collection: 27

1. id=5497cfb4-cf74-40a2-814e-32c123654b8e
   topic=space | doc_number=None
   content=Rockets work by expelling gas at high speed, generating thrust through Newton's ...

2. id=98d411b0-663f-4766-bd0f-a5022f1b060a
   topic=space | doc_number=None
   content=The International Space Station orbits Earth at about 400 km altitude and travel...

3. id=4d794132-aac0-4472-8026-abb21d4c1657
   topic=space | doc_number=None
   content=Spacecraft use gravitational slingshots around planets to gain speed without bur...

4. id=fd5e617f-c5e6-4397-aa0f-c60463c3f81d
   topic=space | doc_number=None
   content=NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from ...

5. id=ac18d533-c202-459c-9753-7fe16e939383
   topic=space | doc_number=None
   content=Solar sails use radiation pressure from sunlight to slowly propel spacecraft wit...

6. id=4d5cfe40-7ad8-4719-ab73-93ad38434131
   topic=biology | doc_number=None
   content=DNA is a double-

## Get Elements by ID's ##

In [29]:
import random
selected_Ids = random.sample(data_chromadb.get("ids"), k=5)
print(f"Randomly selected document IDs for preview: {selected_Ids}")

Randomly selected document IDs for preview: ['01da1442-b2a1-46e6-a53e-96fd4b19d031', 'd61ada04-86a5-4cf6-9813-7a937dc1f0ab', '30f6c584-f823-44e2-af7f-555ec571699d', 'e5c3a6a4-eade-4b49-bde7-a45275b14195', '4d794132-aac0-4472-8026-abb21d4c1657']


In [30]:
selected_documents = data_chromadb.get_by_ids(ids=selected_Ids)
print_stored_documents(f"Randomly selected documents for preview: {selected_documents}")

AttributeError: 'dict' object has no attribute 'get_by_ids'

In [35]:
chroma_retriever = Chroma(collection_name=collection_name, client=client)
query = "What is a Red Bull?"
retrieved_docs = chroma_retriever.similarity_search_with_score(query)
print(f"Retrieved {len(retrieved_docs)} documents for the query: '{query}'")
for index, (doc, score) in enumerate(retrieved_docs, start=1):
    print(f"{index}. {preview_text(doc.page_content)} (Score: {score})")
    

Retrieved 4 documents for the query: 'What is a Red Bull?'
1. The permafrost in Siberia contains vast amounts of methane that could be release... (Score: 1.6580455303192139)
2. Carbon capture technology removes CO2 from the atmosphere and stores it undergro... (Score: 1.7762587070465088)
3. Gradient descent is the core optimisation technique in deep learning, guiding we... (Score: 1.8514325618743896)
4. Rockets work by expelling gas at high speed, generating thrust through Newton's ... (Score: 1.8564975261688232)
